# Block 2: Predictive Maintenance Pipeline
## Scikit-Learn Pipelines & MLflow Experiment Tracking

**Szenario**: Sie sind Data Scientist bei einem Maschinenbauer. Ziel: Maschinenausfälle vorhersagen.  
**Datensatz**: AI4I 2020 Predictive Maintenance (UCI)

---

## 0. Daten laden

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import urllib.request

data_dir = pathlib.Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
csv_path = data_dir / "predictive_maintenance.csv"

if not csv_path.exists():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
    print(f"Lade Datensatz von UCI...")
    try:
        urllib.request.urlretrieve(url, csv_path)
        print(f"Heruntergeladen: {csv_path}")
    except Exception as e:
        print(f"Download fehlgeschlagen: {e}")
        print("Alternative: https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset")
        print(f"CSV manuell speichern als: {csv_path.resolve()}")
else:
    print(f"Datensatz vorhanden: {csv_path}")

df = pd.read_csv(csv_path)
print(f"\nShape: {df.shape[0]:,} Zeilen x {df.shape[1]} Spalten")
df.head()

---
## Aufgabe A: Business Understanding

### A.1 Kosten-Asymmetrie

In der Praxis sind die Kosten von Fehlentscheidungen fast nie symmetrisch. Bei der Vorhersage von Maschinenausfällen gibt es zwei Fehlerarten:

- **False Negative (FN):** Das Modell sagt "kein Ausfall", aber die Maschine fällt aus. → **Ungeplanter Stillstand.** Folgen: Produktionsausfall, Eilreparatur, Lieferverzögerungen, ggf. Folgeschäden an anderen Komponenten. Kosten können schnell im fünf- bis sechsstelligen Bereich liegen.

- **False Positive (FP):** Das Modell sagt "Ausfall", aber die Maschine läuft einwandfrei. → **Unnötige Wartung.** Folgen: Geplanter Stillstand für Inspektion, Personalkosten, ggf. Ersatzteile. Kosten typischerweise im drei- bis vierstelligen Bereich.

Diese **Kosten-Asymmetrie** bestimmt, welche Metrik Sie optimieren sollten. Wenn FN 10× teurer ist als FP, dann ist es besser, lieber einmal zu viel zu warnen als einen Ausfall zu übersehen.

**Schätzen Sie für Ihr Szenario:** Was kostet ein verpasster Ausfall vs. eine unnötige Wartung?

*Ihre Einschätzung*:

- False Negative (Ausfall übersehen): ...
- False Positive (unnötige Wartung): ...
- Verhältnis (ca.): ...

### A.2 Ziel-Metrik

Aus der Kosten-Asymmetrie ergibt sich, welche Metrik für Ihr Modell am wichtigsten ist:

| Metrik | Bedeutung | Wann priorisieren? |
|--------|-----------|-------------------|
| **Accuracy** | Anteil korrekt klassifizierter Fälle | Nur bei balancierten Klassen sinnvoll |
| **Precision** | "Wenn ich warne — wie oft stimmt es?" | Wenn FP teuer ist (z.B. unnötige OP) |
| **Recall** | "Von allen echten Ausfällen — wie viele erkenne ich?" | Wenn FN teuer ist (z.B. Maschinenausfall) |
| **F1-Score** | Harmonisches Mittel aus Precision und Recall | Wenn beide Fehlerarten ähnlich kosten |

**Ihre Aufgabe:** Welche Metrik wählen Sie? Ab welchem Wert ist das Modell nützlich?

*Ihre Ziel-Metrik*: ...

*Ihr Schwellenwert*: ...

*Begründung*: ...

---
## Aufgabe B: EDA & Datenentscheidungen

### B.1 Explorative Analyse

Bevor Sie ein Modell bauen, müssen Sie Ihre Daten verstehen. Achten Sie bei der EDA besonders auf:

- **Klassenverteilung:** Wie viele Ausfälle gibt es? Bei starker Imbalance (z.B. 3% Ausfälle) ist Accuracy als Metrik irreführend — ein Modell das immer "kein Ausfall" sagt, hätte 97% Accuracy!
- **Sensormuster:** Unterscheiden sich die Sensorwerte bei Ausfall vs. kein Ausfall? Boxplots zeigen Ihnen, welche Features informativ sein könnten.
- **Korrelationen:** Welche Features hängen mit dem Target zusammen? Aber Vorsicht: Korrelation ≠ Kausalität.
- **Ausreißer:** Gibt es extreme Werte? Bei Maschinenausfällen sind Extremwerte oft genau das Signal, das Sie suchen — nicht unbedingt Datenfehler.

In [ ]:
# Überblick
print("=== Datentypen ===")
print(df.dtypes)
print(f"\n=== Fehlende Werte ===")
print(df.isnull().sum())
print(f"\n=== Spalten ===")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}, {df[col].nunique()} unique")

In [ ]:
# Klassenverhältnis
target_col = 'Machine failure'
print("=== Klassenverhältnis ===")
print(df[target_col].value_counts())
print(f"\nAusfallrate: {df[target_col].mean()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[target_col].value_counts().plot.bar(ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Machine Failure Verteilung')
axes[0].set_xticklabels(['Kein Ausfall', 'Ausfall'], rotation=0)

# Ausfalltypen
failure_types = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
existing_ft = [c for c in failure_types if c in df.columns]
if existing_ft:
    df[existing_ft].sum().plot.bar(ax=axes[1], color='coral')
    axes[1].set_title('Ausfalltypen')
plt.tight_layout()
plt.show()

TWF – Tool Wear Failure (Werkzeugverschleiß-Ausfall): Das Werkzeug wird zwischen 200 und 240 Minuten Nutzungsdauer zufällig ersetzt oder fällt aus.
HDF – Heat Dissipation Failure (Wärmeabfuhr-Ausfall): Tritt ein, wenn die Differenz zwischen Prozess- und Lufttemperatur unter 8,6 K liegt und die Rotationsgeschwindigkeit unter 1380 U/min.
PWF – Power Failure (Leistungs-Ausfall): Das Produkt aus Drehmoment und Drehzahl (also die Leistung in Watt) liegt außerhalb des Bereichs von 3500–9000 W.
OSF – Overstrain Failure (Überlastungs-Ausfall): Tritt ein, wenn das Produkt aus Werkzeugverschleiß und Drehmoment bestimmte Schwellen überschreitet (abhängig von der Produktqualität L/M/H).
RNF – Random Failures (Zufällige Ausfälle): Jeder Prozess hat eine Wahrscheinlichkeit von 0,1 %, unabhängig von den Parametern auszufallen.


In [ ]:
# Sensorwerte nach Ausfall/kein Ausfall
sensor_cols = [c for c in df.select_dtypes(include='number').columns 
               if c not in [target_col, 'UDI'] + failure_types]

fig, axes = plt.subplots(1, len(sensor_cols), figsize=(4*len(sensor_cols), 4))
if len(sensor_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, sensor_cols):
    df.boxplot(column=col, by=target_col, ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Machine Failure')
plt.suptitle('Sensorwerte nach Ausfallstatus', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Korrelationen
num_df = df[sensor_cols + [target_col]]
corr = num_df.corr()[target_col].drop(target_col).sort_values(ascending=False)
print("Korrelation mit Machine Failure:")
for feat, val in corr.items():
    print(f"  {feat}: {val:.3f}")

### B.2 Drei Entscheidungen

**Entscheidung 1: Ausreißer** -- Entfernen, cappen oder beibehalten?

*Ihre Entscheidung*: ...

*Begründung*: ...

**Entscheidung 2: Product ID** -- Feature oder nicht?

*Ihre Entscheidung*: ...

*Begründung*: ...

**Entscheidung 3: Binär oder Multiclass?**

*Ihre Entscheidung*: ...

*Begründung*: ...

---
## Aufgabe C: Scikit-Learn Pipeline bauen

### C.1 Features und Target definieren

In [ ]:
from sklearn.model_selection import train_test_split

# DEIN CODE: Features und Target definieren
# Überlege: Welche Spalten sind Features? Welche droppen?
# Tipp: 'UDI', 'Product ID' und die Ausfalltyp-Spalten sind keine Features

drop_cols = []  # TODO: Spalten die raus sollen

# X = ...
# y = ...



<details>
<summary><b>Hint C.1</b> (klicke zum Aufklappen)</summary>

**Welche Spalten droppen?**
- `UDI` ist ein laufender Index → kein Feature
- `Product ID` hat 10.000 einzigartige Werte → kein nützliches Feature, hohes Overfitting-Risiko
- Die Ausfalltyp-Spalten (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) sind *Ergebnisse* des Ausfalls, keine Prädiktoren → **Target Leakage!** (In der Praxis wüssten Sie diese Werte nicht im Voraus.)

**Relevante Doku:**
- [`pandas.DataFrame.drop()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html) — `columns=` Parameter für Spaltenliste
- [`sklearn.model_selection.train_test_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) — Achten Sie auf den `stratify`-Parameter bei unbalancierten Klassen!

</details>

### C.2 ColumnTransformer + Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# DEIN CODE: ColumnTransformer bauen
# Numerische Features: Imputer + Scaler
# Kategorische Features (Type): OneHotEncoder

# num_cols = ...
# cat_cols = ...

# preprocessor = ColumnTransformer([
#     ("num", Pipeline([...]), num_cols),
#     ("cat", Pipeline([...]), cat_cols),
# ])


<details>
<summary><b>Hint C.2</b> (klicke zum Aufklappen)</summary>

**Schritt-für-Schritt-Ansatz:**
1. Identifizieren Sie numerische und kategorische Spalten mit `df.select_dtypes()`
2. Bauen Sie für jeden Typ eine eigene Sub-Pipeline (z.B. Imputer → Scaler für numerisch, Imputer → Encoder für kategorisch)
3. Kombinieren Sie beides in einem `ColumnTransformer`

**Relevante Doku:**
- [`sklearn.compose.ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) — Abschnitt "Examples" zeigt das Pattern
- [`sklearn.pipeline.Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) — Mehrere Schritte verketten
- [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [`sklearn.preprocessing.OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) — `handle_unknown='ignore'` für robuste Pipelines

**Wichtig:** `sparse_output=False` beim OneHotEncoder setzen, sonst gibt es Probleme mit manchen Modellen.

</details>

### C.3 Drei Modelle vergleichen

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# DEIN CODE: 3 Modelle als Pipelines definieren und mit CV vergleichen



### C.4 Ergebnistabelle

| Modell | CV-Score (Mean) | CV-Score (Std) | Ihre Einschätzung |
|--------|----------------|----------------|-------------------|
| LogReg | | | |
| RandomForest | | | |
| GradientBoosting | | | |

---
## Aufgabe D: MLflow Tracking

Starten Sie MLflow im Terminal:
```bash
 mlflow ui --backend-store-uri sqlite:///<db_path> --default-artifact-root file://<mlruns_path>  # Siehe nächste Zelle
```

### D.1 Setup

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# SQLite-Backend (nötig für den Overview-Tab der MLflow-UI)
# Artefakte (Modelle, Plots) liegen weiter im mlruns/-Ordner
import os
db_path = os.path.join(os.getcwd(), "mlflow.db")
artifacts_path = os.path.join(os.getcwd(), "mlruns")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("Block2-Predictive-Maintenance")
print(f"MLflow konfiguriert. Tracking DB: {db_path}")
print(f"Artifacts:                       {artifacts_path}")
print(f"\nUI starten mit:")
print(f"  mlflow ui --backend-store-uri sqlite:///{db_path} --default-artifact-root file://{artifacts_path}")

### D.2 Alle Modelle loggen

In [ ]:
# DEIN CODE: Modelle trainieren und in MLflow loggen
# Für jedes Modell einen eigenen Run erstellen und loggen.
# Schaue in die MLflow Doku für die relevanten Funktionen.


<details>
<summary><b>Hint D.2</b> (klicke zum Aufklappen)</summary>

**Workflow pro Modell:**
1. `mlflow.start_run(run_name=...)` als Context Manager (`with`-Block)
2. Pipeline fitten → Predictions auf Testset
3. Metriken berechnen und mit `mlflow.log_metric()` loggen
4. Hyperparameter mit `mlflow.log_param()` loggen
5. Confusion Matrix als Plot speichern → `mlflow.log_artifact()` für die Bilddatei
6. Modell mit `mlflow.sklearn.log_model()` speichern

**Relevante Doku:**
- [MLflow Tracking Quickstart](https://mlflow.org/docs/latest/getting-started/intro-quickstart/index.html)
- [`mlflow.log_param()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_param)
- [`mlflow.log_metric()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_metric)
- [`mlflow.log_artifact()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_artifact)
- [`mlflow.sklearn.log_model()`](https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html#mlflow.sklearn.log_model)
- [`ConfusionMatrixDisplay.from_predictions()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html)

**Tipp:** Speichern Sie die Confusion Matrix mit `fig.savefig("cm_name.png")` und loggen Sie die Datei dann als Artifact.

</details>

### D.3 MLflow UI interpretieren

Öffnen Sie http://localhost:5000

**AUFGABE**: Welches Modell wählen Sie? Warum?

(Fügen Sie hier einen Screenshot der MLflow Vergleichsansicht ein)

*Ihr gewähltes Modell*: ...

*Begründung*: ...

---
## Aufgabe E: Reflexionsfragen

**Frage 1**: Ihr bestes Modell hat Recall 0.85 -- 15% der Ausfälle werden übersehen. Deployen?

*Ihre Antwort*: ...

**Frage 2**: Was passiert wenn Sie den Scaler NICHT in die Pipeline packen?

*Ihre Antwort*: ...

**Frage 3**: In welcher CRISP-DM Phase sind Sie? Was fehlt noch?

*Ihre Antwort*: ...

---
## Bonus: Erreicht Ihr Modell Ihre Ziel-Metrik aus A.2?

Vergleichen Sie Ihr Ergebnis mit dem Schwellenwert, den Sie zu Beginn definiert haben.

*Ziel-Metrik war*: ...

*Erreicht*: ... / ...

*Fazit*: ...